# CNC-AI Benchmark Evaluation
This notebook loads `cnc_questions.jsonl`, runs them through Qwen2.5-Coder-3B-Instruct, and saves the answers to `qwen_results.jsonl`.

In [ ]:
!pip install -q -U transformers accelerate

In [ ]:
import torch
from transformers import pipeline

model_id = "Qwen/Qwen2.5-Coder-3B-Instruct"

print("Loading model...")
pipe = pipeline(
    "text-generation",
    model=model_id,
    torch_dtype="auto",
    device_map="auto"
)
print("Model loaded successfully!")

In [ ]:
import json

# 1. Upload your cnc_questions.jsonl to the Colab files area before running this
input_file = "cnc_questions.jsonl"
output_file = "qwen_results.jsonl"

results = []

print("Starting evaluation...")
with open(input_file, "r") as f:
    for line in f:
        data = json.loads(line)
        
        # Format the prompt
        sys_prompt = "You are an expert CNC programmer and machinist. Answer the following question about CNC machining and G-code. Be concise."
        user_prompt = f"Context:\n{data['context']}\n\nQuestion:\n{data['question']}"
        
        messages = [
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": user_prompt}
        ]
        
        # Run the model
        outputs = pipe(messages, max_new_tokens=256, temperature=0.1)
        answer = outputs[0]["generated_text"][-1]["content"]
        
        # Save result
        data["model_answer"] = answer
        results.append(data)
        
        print(f"Processed {data['id']}: {data['category']}")

# 2. Save the results
with open(output_file, "w") as f:
    for res in results:
        f.write(json.dumps(res) + "\n")
        
print(f"\nDone! Download {output_file} from the Colab files menu to grade the answers.")